# 02 – Tests des stratégies d'anonymisation Presidio

Comparaison des 3 stratégies sur un échantillon de 50 exemples EN et 50 exemples FR.

| Stratégie | Description | Exemple |
|---|---|---|
| `replace` | Remplace par `<TYPE>` | `Jean Dupont` → `<PERSON>` |
| `mask` | Masque les caractères | `Jean Dupont` → `****` |
| `redact` | Supprime complètement | `Jean Dupont` → `` |

In [ ]:
import json
import random
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..')))

from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

NORM_DIR = Path('../data/interim/normalized')
ENTITIES = ['PERSON', 'LOCATION', 'DATE_TIME', 'PHONE_NUMBER', 'EMAIL_ADDRESS', 'NRP']
SCORE_THRESHOLD = 0.70

def build_engines(language):
    model = 'fr_core_news_md' if language == 'fr' else 'en_core_web_lg'
    provider = NlpEngineProvider(nlp_configuration={
        'nlp_engine_name': 'spacy',
        'models': [{'lang_code': language, 'model_name': model}],
    })
    analyzer = AnalyzerEngine(nlp_engine=provider.create_engine(), supported_languages=[language])
    anonymizer = AnonymizerEngine()
    return analyzer, anonymizer

print('Moteurs chargés.')

## 1. Chargement de l'échantillon

In [ ]:
random.seed(42)

def load_sample(file_path, n, language):
    records = []
    with open(file_path, encoding='utf-8') as f:
        for line in f:
            r = json.loads(line)
            if r.get('language') == language:
                records.append(r)
    return random.sample(records, min(n, len(records)))

sample_en = load_sample(NORM_DIR / 'mediqa_normalized.jsonl', 50, 'en')
sample_fr = load_sample(NORM_DIR / 'frenchmedmcqa_normalized.jsonl', 50, 'fr')

print(f'Échantillon EN : {len(sample_en)} exemples')
print(f'Échantillon FR : {len(sample_fr)} exemples')

## 2. Comparaison des 3 stratégies

In [ ]:
def anonymize_with_strategy(text, analyzer, anonymizer, language, strategy_name, params):
    results = analyzer.analyze(text=text, language=language,
                                entities=ENTITIES, score_threshold=SCORE_THRESHOLD)
    if not results:
        return text, 0
    operators = {e: OperatorConfig(strategy_name, params) for e in ENTITIES}
    anonymized = anonymizer.anonymize(text=text, analyzer_results=results, operators=operators)
    return anonymized.text, len(results)

strategies = {
    'replace': ('replace', {'new_value': '<ENTITY>'}),
    'mask':    ('mask',    {'masking_char': '*', 'chars_to_mask': 6, 'from_end': False}),
    'redact':  ('redact',  {}),
}

# Tester sur 5 exemples EN
analyzer_en, anonymizer_en = build_engines('en')
print('=== Exemples EN ===\n')
for record in sample_en[:5]:
    text = record['instruction'][:200]
    print(f'ORIGINAL : {text}')
    for strat_name, (op, params) in strategies.items():
        result, n = anonymize_with_strategy(text, analyzer_en, anonymizer_en, 'en', op, params)
        print(f'{strat_name:8s} [{n} entités] : {result[:200]}')
    print()

In [ ]:
# Tester sur 5 exemples FR
analyzer_fr, anonymizer_fr = build_engines('fr')
print('=== Exemples FR ===\n')
for record in sample_fr[:5]:
    text = record['instruction'][:200]
    print(f'ORIGINAL : {text}')
    for strat_name, (op, params) in strategies.items():
        result, n = anonymize_with_strategy(text, analyzer_fr, anonymizer_fr, 'fr', op, params)
        print(f'{strat_name:8s} [{n} entités] : {result[:200]}')
    print()

## 3. Taux de détection par stratégie

In [ ]:
def count_entities(records, analyzer, language):
    total_entities = 0
    records_with_entities = 0
    entity_type_counts = {}
    for r in records:
        text = r.get('instruction', '') + ' ' + r.get('response', '')
        results = analyzer.analyze(text=text, language=language,
                                    entities=ENTITIES, score_threshold=SCORE_THRESHOLD)
        if results:
            records_with_entities += 1
            total_entities += len(results)
            for res in results:
                entity_type_counts[res.entity_type] = entity_type_counts.get(res.entity_type, 0) + 1
    return total_entities, records_with_entities, entity_type_counts

print('=== Analyse EN ===')
n_en, rec_en, types_en = count_entities(sample_en, analyzer_en, 'en')
print(f'Entités détectées : {n_en} sur {len(sample_en)} enregistrements ({rec_en} affectés)')
for etype, count in sorted(types_en.items(), key=lambda x: -x[1]):
    print(f'  {etype:<20s}: {count}')

print('\n=== Analyse FR ===')
n_fr, rec_fr, types_fr = count_entities(sample_fr, analyzer_fr, 'fr')
print(f'Entités détectées : {n_fr} sur {len(sample_fr)} enregistrements ({rec_fr} affectés)')
for etype, count in sorted(types_fr.items(), key=lambda x: -x[1]):
    print(f'  {etype:<20s}: {count}')

## 4. Conclusion — Choix de la stratégie

**Stratégie retenue : `replace`**

| Critère | `replace` | `mask` | `redact` |
|---|---|---|---|
| Lisibilité post-anonymisation | ✅ bonne | ⚠️ difficile | ❌ perte de contexte |
| Auditabilité | ✅ type visible | ❌ illisible | ❌ supprimé |
| Structure syntaxique préservée | ✅ oui | ⚠️ partielle | ❌ non |
| Utilité pour SFT | ✅ bonne | ⚠️ moyenne | ❌ mauvaise |

`replace` préserve la structure de la phrase (`"Le patient <PERSON> âgé de <DATE_TIME>..."`) tout en rendant l'anonymisation auditable. C'est le meilleur compromis pour un dataset de fine-tuning médical.